# Sham — CPU Autonomous Research Track (real web crawl -> filter -> train)

**What this notebook is (owner, 2026-09-20):** a third, independent training track dedicated to self-directed learning from the live web — real search, real fetch, real safety+MinHash-dedup+perplexity filtering, real training, all in one repeating cycle (`autonomous_pipeline.py`, already verified end-to-end). CPU-only, so it is **not subject to Kaggle's 30 GPU-hours/week quota**, and because it has no accelerator, Kaggle's own native **"Schedule this notebook"** feature accepts it directly (only GPU-attached notebooks are refused).

**Separate checkpoint lineage, same reasoning as sham_cpu_training_track.ipynb:** this track publishes to its own `sham-research-track-checkpoint` dataset, never into the GPU track's `nova-small-checkpoint`. Two independently-scheduled processes training the same weights without coordination would race and could silently regress each other's progress.

**One-time fork from the GPU track (manual, on purpose):** attach the GPU track's `nova-small-checkpoint` dataset as an Input for this notebook's FIRST run only, to start from real trained weights instead of random init; remove it after that and only keep this track's own dataset attached from then on — same convention as the CPU training track.

**One-time setup checklist:**
1. Accelerator: **None**. Internet: **On**.
2. Secrets: `GITHUB_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY` (same three already in use elsewhere).
3. Create an empty Kaggle Dataset named `sham-research-track-checkpoint` once.
4. First run only: Add Input → the GPU track's `nova-small-checkpoint` dataset.
5. Save, then Kaggle's own **"Schedule this notebook"**.

**Real, honest limits, stated plainly (matching data_acquisition.py's and web_access.py's own documented boundaries):** DuckDuckGo search has been observed blocking scraping from some shared cloud IPs before (this project hit that from Render specifically) — if a scheduled run logs zero crawl results repeatedly, check the run's own log output first; `real_search()` already degrades gracefully (empty results, not a crash) and logs the reason rather than failing silently.

### 1) Clone the real code from GitHub

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py"))
sys.path.insert(0, CODE_DIR)
print("Sham code ready at:", CODE_DIR)

In [ ]:
subprocess.run(["pip", "install", "-q", "ddgs"], check=True)
try:
    import tokenizers
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
print("dependencies ready.")

### 2) Model + tokenizer -- resume from THIS track's own checkpoint, or the GPU track's (one-time fork only)

In [ ]:
from pathlib import Path
from model import ShamSmallConfig, ShamSmall, TOTAL_VOCAB_SIZE
from text_tokenizer import train_text_tokenizer, ShamTextTokenizer
from checkpoint import load_checkpoint

SEQ_LEN = 512
device = "cpu"

previous_tokenizer_files = list(Path("/kaggle/input").rglob("*tokenizer*.json"))
if previous_tokenizer_files:
    tokenizer = ShamTextTokenizer.load(str(previous_tokenizer_files[0]))
    print(f"reused existing tokenizer: {previous_tokenizer_files[0]}")
else:
    # A true first run with no prior tokenizer anywhere -- bootstrap one
    # from a tiny real seed corpus so training can start; the crawler's
    # own real content will dominate this track's vocabulary over
    # subsequent scheduled runs regardless.
    import tempfile
    with tempfile.TemporaryDirectory() as _d:
        seed_path = Path(_d) / "seed.txt"
        seed_path.write_text("Sham is a real, from-scratch multimodal transformer. " * 200, encoding="utf-8")
        tokenizer = train_text_tokenizer([str(seed_path)], vocab_size=4000)
    print("trained a fresh bootstrap tokenizer (no previous one found).")
tokenizer.save("/kaggle/working/sham_research_tokenizer.json")

model_cfg = ShamSmallConfig(
    vocab_size=TOTAL_VOCAB_SIZE, d_model=768, n_layers=12, n_heads=12, n_kv_heads=4,
    mlp_hidden=2048, max_seq_len=SEQ_LEN, use_gradient_checkpointing=True,
)
model = ShamSmall(model_cfg)

start_step = 0
previous_checkpoints = sorted(
    Path("/kaggle/input").rglob("step_*.pt"),
    key=lambda p: int(p.stem.split("_")[1]),
)
if previous_checkpoints:
    last_ckpt = previous_checkpoints[-1]
    model, start_step, _ = load_checkpoint(last_ckpt, map_location=device)
    print(f"resumed from: {last_ckpt} (step {start_step:,})")
else:
    print("no previous checkpoint found -- starting from scratch (expected only on a true first run).")

print(f"model: {model.count_parameters():,} real parameters, device={device}")

### 3) Real topics to research this run

One topic per domain, matching the same broad-coverage rotation idea `ai-system/scripts/gather_knowledge.py` already uses -- real breadth every run instead of narrowing onto one area.

In [ ]:
import datetime

_TOPICS_BY_DOMAIN_AR = [
    "أحدث تطورات النماذج اللغوية مفتوحة المصدر",
    "تقنيات تحسين كفاءة تدريب الشبكات العصبية",
    "أفضل ممارسات معالجة اللغة العربية آلياً",
    "مصادر بيانات نصية عربية مفتوحة الرخصة",
    "تقنيات ضغط النماذج وتسريع الاستدلال",
]
day = datetime.date.today().timetuple().tm_yday
topics_today = [_TOPICS_BY_DOMAIN_AR[day % len(_TOPICS_BY_DOMAIN_AR)]]
topics_by_language = {"ar": topics_today}
print("topics this run:", topics_by_language)

### 4) The real autonomous cycle: crawl -> filter -> train, bounded to this session's time budget

In [ ]:
import time
from autonomous_knowledge_crawler import KnowledgeCorpus
from perplexity_filter import PerplexityFilter
from train_from_stream import StreamTrainConfig
from autonomous_pipeline import run_autonomous_cycle

MAX_SESSION_HOURS = 8.5
corpus = KnowledgeCorpus("/kaggle/working/research_corpus")
perplexity_filter = PerplexityFilter(model, tokenizer, device=device)

train_cfg = StreamTrainConfig(
    seq_len=SEQ_LEN,
    batch_size=4,
    lr=3e-4,
    warmup_steps=20,
    total_steps=10_000,  # an upper bound only -- the real stopping point is whatever the crawled corpus + time budget actually allow
    checkpoint_dir="/kaggle/working/checkpoints",
    checkpoint_every=100,
    log_every=10,
)

session_start = time.time()
cycle_results = []
while (time.time() - session_start) < MAX_SESSION_HOURS * 3600:
    result = run_autonomous_cycle(
        model, tokenizer, corpus, topics_by_language, train_cfg,
        perplexity_filter=perplexity_filter, device=device,
        checkpoint_path="/kaggle/working/checkpoints/final.pt",
    )
    cycle_results.append(result)
    print(f"cycle {len(cycle_results)}: crawled+added={result.crawl_stats.added}, "
          f"trained {len(result.train_losses)} real steps, corpus size={result.total_documents_in_corpus}")
    if not result.train_losses:
        # Nothing new AND nothing left to train on this cycle (e.g. a
        # search provider hiccup with an already-exhausted corpus) --
        # pause briefly rather than spin a tight, pointless loop for
        # the rest of the session budget.
        time.sleep(60)

final_step = start_step + sum(len(r.train_losses) for r in cycle_results)
print(f"\nsession done: {len(cycle_results)} real cycles, final step {final_step:,}")

### 5) Publish this track's own checkpoint (its own dataset, never the GPU track's)

In [ ]:
import json as _json
import shutil as _shutil

subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=False)

KAGGLE_USERNAME = UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY")
# "-v2": never let this track's first-ever publish be the first time we
# discover whether its dataset name accepts "-r zip" -- start on a name
# Kaggle has never seen, so its first version establishes zip-mode
# compatibility from birth (same reasoning as the GPU and CPU-training
# tracks' identical fix, 2026-09-20).
DATASET_SLUG = f"{KAGGLE_USERNAME}/sham-research-track-checkpoint-v2"

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    _shutil.rmtree(upload_dir)
(upload_dir / "checkpoints").mkdir(parents=True)
for ckpt in Path("/kaggle/working/checkpoints").glob("*.pt"):
    _shutil.copy2(ckpt, upload_dir / "checkpoints" / ckpt.name)
_shutil.copy2("/kaggle/working/sham_research_tokenizer.json", upload_dir / "sham_research_tokenizer.json")

metadata = {"title": "sham-research-track-checkpoint-v2", "id": DATASET_SLUG, "licenses": [{"name": "unknown"}]}
(upload_dir / "dataset-metadata.json").write_text(_json.dumps(metadata))

# Deterministic existence check -- ask Kaggle directly whether this
# dataset is already in the account's own list, instead of guessing from
# an error message's exact wording (not a stable contract to parse).
_list_result = subprocess.run(["kaggle", "datasets", "list", "-m", "--csv"], capture_output=True, text=True)
_dataset_exists = DATASET_SLUG in (_list_result.stdout or "")

# "-r zip": "-r skip" (Kaggle CLI's documented default) silently IGNORES
# subdirectories instead of uploading them -- checkpoints/ is one, so a
# version published under "-r skip" never actually contains the .pt
# files. "-r zip" actually uploads it, as a .zip archive.
if _dataset_exists:
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", f"research-track auto-update at step {final_step:,}", "-r", "zip"],
        capture_output=True, text=True,
    )
else:
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(upload_dir), "-r", "zip"],
        capture_output=True, text=True,
    )
_combined = (result.stdout or "") + (result.stderr or "")

if result.returncode == 0 and "error" not in _combined.lower():
    verb = "published to" if _dataset_exists else "created"
    print(f"{verb} {DATASET_SLUG} at step {final_step:,} -- the next scheduled run will pick it up automatically.")
elif "incompatible" in _combined.lower():
    print(
        "WARNING: this dataset was created under an incompatible upload mode. No code-level fix for it "
        "specifically -- bump the -v2 suffix above to -v3 (a name Kaggle has never seen) and re-run. "
        "The checkpoint itself is still safe in this session's own Output regardless."
    )
    print(_combined)
else:
    print("WARNING: failed to publish -- the checkpoint is still safe in this session's own Output. "
          "Check that KAGGLE_USERNAME/KAGGLE_KEY are correct.")
    print(_combined)

### 6) Report this session to the owner on Telegram (optional)

Same idea as Nova's automatic weekly Telegram report, applied to this track's own real crawl+train cycles -- see `telegram_report.py`. Skips cleanly if `TELEGRAM_BOT_TOKEN`/`TELEGRAM_CHAT_ID` secrets aren't configured (see the cell below for one-time setup).

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    TELEGRAM_BOT_TOKEN = UserSecretsClient().get_secret("TELEGRAM_BOT_TOKEN")
    TELEGRAM_CHAT_ID = UserSecretsClient().get_secret("TELEGRAM_CHAT_ID")
except Exception:
    # Both secrets are OPTIONAL -- a real, one-time setup step (owner
    # request, 2026-09-21, comparing Nova's automatic Telegram reports
    # against Sham's total silence between manual Kaggle-log checks).
    # Add-ons -> Secrets -> add TELEGRAM_BOT_TOKEN (an existing bot's
    # token) and TELEGRAM_CHAT_ID (the owner's own Telegram numeric id,
    # e.g. the SUPER_ADMIN_TELEGRAM_ID already used elsewhere in this
    # project) to receive this. Not configured -> report is skipped,
    # never blocks or fails the session itself.
    TELEGRAM_BOT_TOKEN, TELEGRAM_CHAT_ID = None, None

from autonomous_pipeline import format_cycle_report
from telegram_report import send_telegram_message

_published_slug = DATASET_SLUG if (result.returncode == 0 and "error" not in _combined.lower()) else None
_report_text = format_cycle_report(cycle_results, "البحث الذاتي المستمر (Track B)", final_step, _published_slug)
print(_report_text)
send_telegram_message(TELEGRAM_BOT_TOKEN, TELEGRAM_CHAT_ID, _report_text)
